In [7]:
!pip install requests beautifulsoup4 lxml

In [8]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from pprint import pprint

In [18]:
def get_soup_from_url(url):

    try:
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        }
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        return BeautifulSoup(response.text, 'html.parser')
    except requests.exceptions.RequestException as e:
        return None

url = 'https://ria.ru/'
soup = get_soup_from_url(url)

In [20]:
if soup:
    title = soup.title.string if soup.title else "Заголовок не найден"
    print(f"ЗАГОЛОВОК СТРАНИЦЫ:")
    print(f"{title}\n")
else:
    print("Нет данных для парсинга")

ЗАГОЛОВОК СТРАНИЦЫ:
РИА Новости - события в Москве, России и мире сегодня: темы дня, фото, видео, инфографика, радио



In [23]:
if soup:
    titles = soup.find_all('span', class_='cell-list__item-title')

    print(f"ЗАГОЛОВКИ НОВОСТЕЙ (количество заголовков: {len(titles)})")

    if titles:
        for i, title in enumerate(titles, 1):
            print(f"{i}. {title.text.strip()}")
    else:
        print("Заголовки не найдены. Возможно, класс изменился.")
else:
    print("Нет данных для парсинга")

ЗАГОЛОВКИ НОВОСТЕЙ (количество заголовков: 22)
1. Жертвами израильского удара по школе в Иране стали 85 детей
2. Путин обсудил ситуацию вокруг Ирана с Совбезом
3. ФРГ, Франция и Британия заявили, что не участвовали в ударах по Ирану
4. ВС России освободили Горькое в Запорожской области
5. Орбан заявил, что Венгрия не поддастся шантажу Зеленского
6. МВД Кубы установило организатора диверсии с катером из США
7. "Четкий сигнал". Происходящее на границе с Россией всполошило Финляндию
8. В США сделали заявление о потоплении американского авианосца Ираном
9. "Закончится катастрофой". Нападение на Иран вызвало панику во Франции
10. По Крымскому мосту возобновили движение
11. "Приезжай в Москву". Слова Зеленского о Путине разозлили Запад
12. В США нашли виновного в начале военной операции против Ирана
13. Заявление Мерца о России вызвало внезапную реакцию в Германии
14. "Будет сложно". СМИ узнали, как операция США в Иране повлияет на Украину
15. Россиянам назвали отечественные альтернативы Tel

In [24]:
if soup:
    links = soup.find_all('a')

    print(f"ССЫЛКИ НА СТРАНИЦЕ (найдено: {len(links)}):")

    links_data = []

    for i, link in enumerate(links, 1):
        href = link.get('href', 'Нет ссылки')
        text = link.text.strip() if link.text else 'Нет текста'

        short_text = text[:50] + "..." if len(text) > 50 else text

        print(f"{i}. Текст: {short_text}")
        print(f"   Ссылка: {href}\n")

        # Сохраняем в список для дальнейшего анализа
        links_data.append({
            'index': i,
            'text': text,
            'href': href
        })

    print(f"Всего {len(links_data)} ссылок")
else:
    print("Нет данных для парсинга")

ССЫЛКИ НА СТРАНИЦЕ (найдено: 205):
1. Текст: 
   Ссылка: https://ria.ru

2. Текст: 
   Ссылка: https://ria.ru/

3. Текст: 
   Ссылка: https://ria.ru/

4. Текст: 
   Ссылка: https://ru.wikipedia.org/wiki/%D0%A0%D0%98%D0%90_%D0%9D%D0%BE%D0%B2%D0%BE%D1%81%D1%82%D0%B8

5. Текст: 
   Ссылка: https://twitter.com/rianru

6. Текст: 
   Ссылка: https://vk.ru/ria

7. Текст: 
   Ссылка: https://ok.ru/ria

8. Текст: 
   Ссылка: https://www.youtube.com/user/rianovosti

9. Текст: 
   Ссылка: https://flipboard.com/@rianovosti

10. Текст: 
   Ссылка: https://zen.yandex.ru/ria

11. Текст: 
   Ссылка: https://twitter.com/riabreakingnews

12. Текст: 
   Ссылка: https://invite.viber.com/?g2=AQAOuCQJzow8L0hQtsB3j3zzc7YMKEHijsslO3ZsCFmFcZGSed0OTOa0ruAXXZn6&lang=ru

13. Текст: 
   Ссылка: https://tgclick.com/rian_ru

14. Текст: 
   Ссылка: https://yandex.ru/maps/org/rossiya_segodnya/1061985604/?ll=37.590466%2C55.737481&z=14

15. Текст: 
   Ссылка: https://www.google.com/maps/place/%D0%A0%D0%98%D0%90+%D0%9D%D

In [27]:
if 'links_data' in locals() and links_data:

    print("АНАЛИЗ СОБРАННЫХ ССЫЛОК:")


    internal_links = []
    external_links = []
    empty_links = []

    for link in links_data:
        href = link['href']
        if href == 'Нет ссылки':
            empty_links.append(link)
        elif href.startswith('http') and 'ria.ru' not in href:
            external_links.append(link)
        else:
            internal_links.append(link)

    print(f"Внутренних ссылок (на ria.ru): {len(internal_links)}")
    print(f"Внешних ссылок: {len(external_links)}")
    print(f"Ссылок без href: {len(empty_links)}")

    if internal_links:
        print("Примеры внутренних ссылок:")
        for link in internal_links[:5]:
            print(f"   - {link['href']}")
else:
    print("Нет данных о ссылках для анализа")

АНАЛИЗ СОБРАННЫХ ССЫЛОК:
Внутренних ссылок (на ria.ru): 145
Внешних ссылок: 32
Ссылок без href: 28
Примеры внутренних ссылок:
   - https://ria.ru
   - https://ria.ru/
   - https://ria.ru/
   - https://cdnn21.img.ria.ru/i/schema_org/ria_logo.png
   - https://cdnn21.img.ria.ru/i/schema_org/ria_logo.png
